In [1]:
print("hello world")

hello world


In [1]:
import os
os.getcwd()

'/home/aman/Desktop/Kidney-Disease-classifier/research'

In [2]:
pwd()

'/home/aman/Desktop/Kidney-Disease-classifier/research'

In [3]:
cd('..')

[Errno 2] No such file or directory: '(..)'
/home/aman/Desktop/Kidney-Disease-classifier/research


In [5]:
os.chdir('../')

In [9]:
%pwd

'/home/aman/Desktop/Kidney-Disease-classifier'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np

ALLOWED_EXTENSIONS = {".jpg", ".jpeg", ".png"}
IMG_SIZE = (28, 28)

x = []
y = []

path = Path("Data")

for class_dir in path.iterdir():

    if not class_dir.is_dir():
        continue

    for image_path in class_dir.iterdir():

        # 1. Extension check
        if image_path.suffix.lower() not in ALLOWED_EXTENSIONS:
            continue

        try:
            # 2. Open image
            with Image.open(image_path) as img:

                # 3. Verify image
                img.verify()

            # 4. Open again because verify() invalidates the image object
            with Image.open(image_path) as img:

                # 5. Check dimensions
                width, height = img.size

                if width < 100 or height < 100:
                    continue

                # 6. Convert to grayscale
                img = img.convert("L")

                # 7. Resize
                img = img.resize(IMG_SIZE)

                # 8. Convert to NumPy
                img = np.array(img, dtype="float32")

                x.append(img)
                y.append(class_dir.name)

        except Exception as e:
            print(f"Skipping {image_path}: {e}")

x = np.array(x, dtype="float32") / 255.0
y = np.array(y)

print(x.shape)
print(y.shape)

In [13]:
os.listdir(Path('Data'))

['Cyst', 'Normal', 'Stone', 'Tumor']

In [16]:
x

['Data/Cyst/Cyst- (2022).jpg',
 'Data/Cyst/Cyst- (588).jpg',
 'Data/Cyst/Cyst- (165).jpg',
 'Data/Cyst/Cyst- (2217).jpg',
 'Data/Cyst/Cyst- (2851).jpg',
 'Data/Cyst/Cyst- (1420).jpg',
 'Data/Cyst/Cyst- (3083).jpg',
 'Data/Cyst/Cyst- (1582).jpg',
 'Data/Cyst/Cyst- (1357).jpg',
 'Data/Cyst/Cyst- (3686).jpg',
 'Data/Cyst/Cyst- (3182).jpg',
 'Data/Cyst/Cyst- (3503).jpg',
 'Data/Cyst/Cyst- (3691).jpg',
 'Data/Cyst/Cyst- (17).jpg',
 'Data/Cyst/Cyst- (2349).jpg',
 'Data/Cyst/Cyst- (3118).jpg',
 'Data/Cyst/Cyst- (1896).jpg',
 'Data/Cyst/Cyst- (3455).jpg',
 'Data/Cyst/Cyst- (695).jpg',
 'Data/Cyst/Cyst- (1685).jpg',
 'Data/Cyst/Cyst- (458).jpg',
 'Data/Cyst/Cyst- (3158).jpg',
 'Data/Cyst/Cyst- (2056).jpg',
 'Data/Cyst/Cyst- (829).jpg',
 'Data/Cyst/Cyst- (2778).jpg',
 'Data/Cyst/Cyst- (3229).jpg',
 'Data/Cyst/Cyst- (1024).jpg',
 'Data/Cyst/Cyst- (448).jpg',
 'Data/Cyst/Cyst- (2119).jpg',
 'Data/Cyst/Cyst- (89).jpg',
 'Data/Cyst/Cyst- (1545).jpg',
 'Data/Cyst/Cyst- (3488).jpg',
 'Data/Cyst/Cyst- 

In [ ]:
! pip install ipykernel

In [6]:
import os
import tensorflow as tf
import keras_tuner as kt
import matplotlib.pyplot as plt

from tensorflow.keras.utils import image_dataset_from_directory, plot_model
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split

ModuleNotFoundError: No module named 'tensorflow.python'

In [ ]:
train_ds = image_dataset_from_directory(
    directory = "/home/aman/Desktop/Kidney-Disease-classifier/Data/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone",
    labels = "inferred",
    label_mode = "categorical",
    image_size = (224,224),
    batch_size = 32,
    color_mode = "grayscale",
    shuffle = True,   
    validation_split = 0.2,
    subset = "Training",
    seed = 123 
)

In [ ]:
train_ds , valid_ds = train_test_split(train_ds, test_size=0.2, random_state=42)

In [ ]:
test_ds = image_dataset_from_directory(
    directory = "/home/aman/Desktop/Kidney-Disease-classifier/Data/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone",
    labels = "inferred",
    label_mode = "categorical",
    image_size = (224,224),
    batch_size = 32,
    color_mode = "grayscale",
    shuffle = True,   
    validation_split = 0.2,
    subset = "validation",
    seed = 123 
)

In [ ]:
model = Sequential()
model.add(Flatten(input_shape=(224,224,1)))
model.add(Dense(256, activation='relu'))
model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(4, activation='softmax'))

In [ ]:
def final_model(hp):
    model = Sequential()

    nums_layers = hp.Int('num_layers', min_value=1, max_value=10, step=1)
    for i in range(nums_layers):
        if i == 0:
            model.add(Flatten(input_shape=(224,224,1)))
        else:
            model.add(Dense(
                units=hp.Int('units_' + str(i), min_value=32, max_value=512, step=32),
                activation = hp.Choice('activation_' + str(i), values=['relu', 'tanh', 'sigmoid'])
            ))

            model.add(Dropout(rate=hp.Float('dropout_' + str(i), min_value=0.0, max_value=0.5, step=0.1)))

    model.add(Dense(4, activation='softmax'))

    model.compile(
        optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop','Nadam']),
        loss = 'categorical_crossentropy',
        metrics = ['accuracy']
    )

    return model




In [ ]:
tuner = kt.Hyperband(
    final_model,
    objective = 'val_accuracy',
    max_trials = 10,
    directory = 'hyperband',
    project_name = 'kidney_disease_classifier'
)

In [ ]:
tuner.search(training_data = train_ds, epochs=10, validation_data=valid_ds)

In [ ]:
model = tuner.get_best_models(num_models=1)[0]

In [ ]:
model.summary()

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5,verbose=1, mode='min',restore_best_weights=True)



In [ ]:
history = model.fit(
    train_ds,
    validation_data = test_ds,
    epochs = 50,
    callbacks = [early_stopping]
)

In [ ]:
plot_model(model, to_file='model.png', show_shapes=True, show_layer_names=True)

In [ ]:
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()